In [ ]:
using Plots, BSON, Flux, cuDNN, CUDA 
import Flux.Zygote
include("utils.jl");

In [ ]:
model_reg = BSON.load("model_regularized.bson")[:model] # model with regularisation
BSON.@load "model_unregularized.bson" model # model without regularisation

In [ ]:
#some possible pair potentials
function Lj(x;ϵ = 1)  #Lennard-Jones
    σ = 1
    if 0 <= x <= 1.5
        return 4*ϵ*((σ/x)^12-(σ/x)^6)
    else
        return 0
    end
end

HR(x) = 0 <= x < 1 ? Inf : 0 #hard rods

function gaussian(x) 
    if 0 <= x
        return 2*exp(-2*x^2)
    else
        return 0
    end
end

function ramp(x;a = 3, b = 1)
    if 0 <= x < 1
        return a-b*x
    else
        return 0
    end
end


In [ ]:
ϕ = get_params(gaussian) # discrete values of the pair potential ϕ
r = collect(0.005:0.01:1.5)
plot(r,ϕ, xlabel = "r/σ", ylabel = "ϕ/ϵ")

In [ ]:
T,µ = 1.5,1; #thermodynamic parameters: temperature and chemical potential

We compare four different routes to obtain the pair distribution function $g(r)$:
    
(1) Meta-OZ-route (by using $\rho$ as integration variable):
### $ g(r) =  \frac{-\beta}{L\rho_b^2} \int_0^{\rho_b}d\rho' \frac{\partial \mu_b(\rho')}{\partial \rho'}  \chi_{\phi}^b(r;\rho')$

In [ ]:
function get_g_MOZ_ρ(µ::Number,T::Number,ϕ::AbstractArray;nsteps = 90, model = model_reg, V0 = x -> 0,L = 4.1,dx=0.01)
    xs,ρb = minimize(L,μ,T,ϕ,V0,model)
    ρs = collect(0:ρb[1]/nsteps:ρb[1])[3:end]
    integrand = []
    dρ = 0.001
    for ρ in ρs
        ∂µ∂ρ = (µb(ρ+dρ,T,ϕ)-µb(ρ-dρ,T,ϕ))/(2*dρ)
        push!(integrand, ∂µ∂ρ*get_χb(Float32.(ρ),Float32.(T), Float32.(ϕ),model))
    end    
    χs = hcat(integrand...)
    G = sum(χs,dims = 2)*(ρs[2]-ρs[1])/T
    -G/(ρb[1]^2),χs,ρs   
end

In [ ]:
g_MOZ_ρ,_,_ = get_g_MOZ_ρ(µ,T,ϕ); #uses the regularized model by default

(2) Meta-OZ-route (by using $\mu$ as integration variable):
### $ g(r) =  \frac{-\beta}{L\rho_b^2} \int_{-\infty}^{\mu}d\mu'\chi_{\phi}^b(r;\rho(\mu'))$

In [ ]:
function get_g_MOZ_µ(µ::Number,T::Number,ϕ::AbstractArray; dµ = 0.025, model = model_reg, V0 = x -> 0, L = 4.1,dx = 0.01)
    µs = collect(-4-(µ - round(µ,digits = 1)):dµ:µ)
    integrand = []
    ρ = zeros(round(Int, L/0.01))
    for µ in µs
        xs, ρ = minimize(L, μ, T, ϕ, V0, model,ρ0 = ρ[1])
        push!(integrand,get_χb(Float32.(ρ[1]),Float32.(T), Float32.(ϕ),model))
    end
    χs = hcat(integrand...)
    G = sum(χs,dims = 2)*dµ/T
    -G/(ρ[1]^2), χs, µs 
end

In [ ]:
g_MOZ_µ,χs, µs = get_g_MOZ_µ(µ,T,ϕ);

(3) DFT test particle minimization: 
### $ \rho_b g(r) = \exp(c_1(r; [\rho_b g,\beta\phi])-\beta\phi(r)+\beta\mu)$

In [ ]:
function get_g_testparticle(μ::Number,T::Number,Φ::Function;model = model_reg, L = 20,dx = 0.01)
    V(x) = Φ(x-L/2) + Φ(-(x-L/2)) #external potential is set to pair potential
    xs, ρ = minimize(L, μ, T, Φ, V, model)
    g = ρ/ρ[1] #devide by ρb
    return xs[1:500], g[Integer(L/(2*dx))+1:Integer(L/(2*dx))+500]
end

In [ ]:
xn,g_tp = get_g_testparticle(µ,T,gaussian); #be sure to use the same potential function as in discrete version

(4) Functional differentation of $ F_{\mathrm{exc}} $ w.r.t $\beta\phi$:
### $g(r)=  \frac{1}{L\rho_b^2} \frac{\delta F_{\mathrm{exc}}[\rho,\beta\phi]}{\delta  \phi (r)}$

In [ ]:
function get_g_δFexcδϕ_finitediff(µ::Number,T::Number,ϕ::AbstractArray; model = model_reg, L::Number = 4.1, Vext::Function = x -> 0)
    xs, ρ = minimize(L, μ, T, ϕ, Vext, model)
    Fexc_func = get_βFexc_funcintegral(xs,model)
    dβϕ = 0.01
    dx = xs[2] - xs[1]
    res = []
    for i=1:150
        βϕ = copy(ϕ)/T
        βϕ[i] = βϕ[i] + dβϕ
        v1 = Fexc_func(ρ,βϕ)
        βϕ[i] = βϕ[i] -2*dβϕ
        v2 = Fexc_func(ρ,βϕ)
        push!(res, (v1-v2)) 
    end
    res / ((ρ[1]^2*L)*(2*dβϕ*dx))
end

In [ ]:
g_Fexϕ = get_g_δFexcδϕ_finitediff(µ,T,ϕ);

In [ ]:
plot(r,g_tp[1:150], label = "test particle",xlabel = "r/σ",ylabel="g")
plot!(r,g_MOZ_ρ, label = "MetaOZ - ρ")
plot!(r,g_MOZ_µ, label = "MetaOZ - µ")
plot!(r,g_Fexϕ, label = "LδFex/δϕ/ρb²L")

In [ ]:
plot_χs(χs,µs,"µ/ϵ") #plot integrated χᵩ for meta-OZ-route

In [ ]:
#check correct normalization of G 
L = 5
Vext(x) = 0
xs,ρ = minimize(L, μ, T, ϕ, Vext, model_reg)
ρb = ρ[1]
check_normalization(g_tp,L,ρb,ϕ,T)